# register_embryos — cohort walkthrough

One cohort, start to finish. A **cohort** is every embryo sharing
`(genotype, timepoint, view, magnification)`; that 4-tuple names the output directory.

The two steps that used to be hand-edited literals — rotation angles and contrast
limits — happen in one widget in step 3, and are saved as JSON so every later run
(including from the CLI) reproduces them without re-deciding anything.

Slow steps are flagged. Each `wf.<step>()` stores its result on `wf`, so a later
step can be re-run with different parameters without repeating the earlier ones.

## 0. Paths and setup

In [ ]:
%matplotlib inline
from datetime import datetime
from pathlib import Path

import register_embryos as re
from register_embryos import CohortWorkflow

ND2_DIR = Path("/net/trapnell/vol1/home/waltno/lpm/data/hcr/wt/wt_lpm_nd2_12s_dorsal")
OUT_ROOT = Path("/net/trapnell/vol1/home/waltno/lpm/data/hcr") / datetime.now().strftime("%y%m%d")

print("register_embryos", re.__version__)
print("open3d ICP backend available:", re.HAS_OPEN3D)
print("filename spec:", re.FILENAME_SPEC)
print("outputs ->", OUT_ROOT)

## 1. What cohorts are in this directory?

Always run this first — it is the cheapest way to catch a misnamed file, which
otherwise shows up as a one-embryo cohort that cannot be registered.

If any filename is not yet on the spec, fix it once with
`register-embryos rename DIR --timepoint 12s --apply` (reversible via the manifest
it writes).

In [ ]:
cohorts = re.scan(ND2_DIR)

## 2. Load — the slow first step

Reads each ND2, max-projects `bin_size` z-slices into each bin, normalises every
channel to [0,1], and reads the voxel size from the file metadata.

Max rather than mean projection: HCR puncta are sparse and bright, and averaging a
punctum over several mostly-empty slices dilutes it below the signal threshold.

The raw `(Z,C,Y,X)` stack is dropped after binning — at ~1.5 GB per embryo a whole
cohort of them will not fit alongside Cellpose.

In [ ]:
wf = CohortWorkflow.from_directory(ND2_DIR, output_root=OUT_ROOT,
                                   cohort="wt_12s_dorsal_20X")

# bin_size depends on how you intend to segment:
#   2D  -> 7   (binning gives each plane enough signal to segment alone)
#   3D  -> 1   (3D takes the WHOLE stack; binning destroys what it works from)
wf.load(bin_size=7)          # SLOW: a few minutes for a 7-embryo cohort

v = wf.volumes[0]
print(f"\n{v.embryo_id}")
print(f"  binned shape (Z_bins, Y, X): {v.shape}")
print(f"  voxel: {v.voxel.xy_um:.3f} x {v.voxel.xy_um:.3f} x {v.voxel.z_um:.3f} um")
print(f"  raw anisotropy    : {v.voxel.anisotropy:.2f}")
print(f"  binned anisotropy : {v.binned_voxel.anisotropy:.2f}")
print(f"  gene map: {v.gene_map}")

## 3. Prepare — rotation and contrast, in one widget

Left panel: the rotated frame. Middle: its histogram with the contrast window
shaded. Right: **what segmentation will actually see**, with saturated/floored
percentages.

- **Orientation** applies to the whole embryo (all channels, all z-bins). Use the
  ±90° nudges to get anterior pointing the same way across the cohort, then the
  slider to fine-tune. A warning appears if rotating inside the fixed 1024×1024
  canvas would clip signal — tick *grow canvas* if so.
- **Contrast** applies to the selected channel only. `Accept contrast` advances to
  the next channel or embryo, so the cohort is walked with one button.

Both are written to `orientation.json` and `contrast_limits.json` in the cohort
directory as you accept them. Re-opening this widget picks up where you left off.

In [ ]:
config = wf.prepare(transform="none")   # transform="log2" lifts dim puncta
config

### No ipywidgets? Same job, non-interactively

`transform="log2"` maps [0,1] onto [0,1] exactly while compressing the bright end,
so faint puncta survive a window that would otherwise clip them.

In [ ]:
# from register_embryos import OrientationSet, auto_contrast_limits, preview_contrast
#
# # the notebook's old `angles = [215, 155, 165, 70]`, but recorded:
# orientations = OrientationSet.from_angles(
#     [v.embryo_id for v in wf.volumes], [215, 155, 165, 70, 0, 0, 0]
# )
# contrast = auto_contrast_limits(wf.volumes, low_percentile=1, high_percentile=99.5)
# preview_contrast(wf.volumes[0], contrast)   # render the chosen limits to a figure

## 4. Bake in the rotation and contrast

`auto_contrast=True` fills anything you did not set from percentiles (p90/p99.9)
rather than raising.

Do not trust it for anything you intend to interpret. An HCR channel is mostly
background — on a real 20× dorsal stack the median normalised intensity is 0.004
and the 99.5th percentile only 0.098 — so a percentile pair chosen badly sweeps
the dim background shoulder into "signal". With the naive `p1/p99.5`, **36–41 % of
pixels landed above the 0.05 threshold and essentially every nucleus read as
expressing**. `auto_contrast_limits` prints the resulting positive fraction per
channel (`[NN%+]`) and warns above 25 %, so check it — and prefer the widget.

In [ ]:
wf.apply_prep(config, auto_contrast=True)

print(f"\n{len(wf.adjusted)} volumes ready for segmentation")
print("history:", wf.adjusted[0].history)

## 5. Segment — the slow second step

| mode | z input | what it does | when |
|---|---|---|---|
| `"2d"` | binned | Cellpose per z-bin | fast, robust; labels are per-slice, so a nucleus spanning two bins becomes two rows |
| `"3d"` | **unbinned** | one pass, `do_3D=True` + voxel anisotropy | labels consistent through z, one row per nucleus with a true 3D centroid — the right input for ICP |
| `"2d+link"` | binned | 2D, then link slices by mask IoU | middle ground, no 3D flow cost |

**3D needs the whole z-stack, unbinned, and this is enforced.** Binning is a
concession for 2D. At `bin_size=7` (1.5 µm z-step → 10.5 µm per plane) a ~6 µm
nucleus spans 0.57 planes, so `do_3D` has nothing to link across z and just runs
slowly. `wf.segment(mode="3d")` raises if the cohort was loaded binned — reload
with `wf.load(bin_size=1)` first, then `apply_prep` again.

Unbinned is also much heavier (~0.8 GB per channel as float32 for a 200-plane
1024² stack, and Cellpose needs several multiples of that), so keep
`max_workers=1` and use a GPU — see `docs/qsub_gpu_segmentation.sh`.

Two separate knobs, often confused:

- **`anisotropy`** — z:xy voxel aspect ratio, from the ND2 × the binning factor.
  Cellpose rescales z with it so a spherical nucleus looks spherical. Too low ⇒
  nuclei split along z; too high ⇒ they merge.
- **`diameter`** — nucleus size in **xy pixels**. `None` lets Cellpose estimate it.

In [ ]:
# 2D on the cohort as loaded above (bin_size=7):
wf.segment(mode="2d", diameter=None, gpu=False, max_workers=2)   # SLOW

for s in wf.segmented:
    print(f"  {s.embryo_id}: {s.n_labels} labels")

In [ ]:
# For 3D, reload unbinned first — then segment on a GPU.
#
# wf.load(bin_size=1)              # SLOW and memory-heavy
# wf.apply_prep(config)
# wf.segment(mode="3d", gpu=True, max_workers=1)

## 6. Assign signal pixels, build the nucleus table

HCR signal sits *around* nuclei, not inside the nuclear stain, so measuring only
inside the Cellpose mask throws most of it away. Each above-threshold pixel is
given to its nearest nucleus, and the per-nucleus value is the mean over that
expanded territory.

Pixels dropped as background become a sentinel (`0.3`), not `0`, and the mean
excludes exactly that sentinel — a pixel that was never measured must not read as
a measured zero, or every nucleus mean gets dragged toward zero in proportion to
how much empty space its territory covers.

In 3D mode the distances are in **micrometres**, so an anisotropic stack does not
preferentially assign along z.

In [ ]:
wf.build_tables(signal_threshold=0.05)

print()
print(wf.combined.head())
print("\ncolumns:", list(wf.combined.columns))

## 7. Register — point-to-point ICP onto a cohort reference

Each embryo is aligned to one reference, so excluding an embryo never changes how
the others land — only the atlas built from them.

Registration downsamples **uniformly in space** first (`isotropic_downsample`):
sampling uniformly at random keeps dense regions dense, and ICP would then fit
those and ignore the sparse ones. Each axis is normalised to [0,1] first so z (a
few dozen bins) is weighted like x and y (a thousand pixels).

A PCA coarse alignment runs before ICP — ICP is a local method, and two embryos
mounted 90° apart will not find each other from a cold start.

Residuals are reported with **the same metric on both sides**. Comparing a mean
nearest-neighbour distance against an RMS one makes a good fit look like a
regression, because RMS of a positive quantity always exceeds its mean.

In [ ]:
wf.register(
    reference_embryo_id=None,      # None = the first embryo in the cohort
    n_downsample=5000,
    max_correspondence_distance=500,   # in xy pixels, calibrated to 1024x1024
    max_iteration=500,
)

print()
print(wf.registration.stats.to_string(index=False))

### Registration QC — always look, do not trust the number alone

A residual cannot reveal an embryo that converged to a plausible-looking wrong
pose. Six panels per embryo: XY/XZ/YZ raw, then XY/XZ/YZ registered, reference in
grey underneath.

In [ ]:
from register_embryos import plot_registration_2d

plot_registration_2d(
    wf.registration.registered, wf.registration.reference_embryo_id,
    mode="light", suptitle=f"{wf.cohort.name} — ICP QC",
);

## 8. Atlas — the composite embryo

At each anchor point (a reference-embryo nucleus), average the **k nearest nuclei
pooled across every embryo** — position and gene intensity both.

Choosing `k` is a real trade-off. With N embryos, `k ≈ N` averages roughly one
nucleus per embryo: it smooths between-embryo variability while preserving spatial
detail. Larger `k` blurs domain boundaries — fine for broad domains, harmful if you
are measuring a domain edge.

`atlas_diagnostics` reports the neighbour radius and how many distinct embryos
actually contributed, which is how you tell whether an atlas point is a consensus
or just one embryo.

In [ ]:
wf.build_atlas(k_neighbors=None,      # None = the embryo count
               n_points=None)         # None = one point per reference nucleus

from register_embryos import atlas_diagnostics
print()
print(atlas_diagnostics(wf.atlas).T.to_string())

In [ ]:
# Leave-one-out variant: no atlas point may draw on its own embryo.
# Use this if you intend to ask how well the reference agrees with the atlas —
# otherwise it is partly predicting itself.
#
# from register_embryos import build_atlas
# loo = build_atlas(wf.registration.registered,
#                   reference_embryo_id=wf.registration.reference_embryo_id,
#                   k_neighbors=4, exclude_self_embryo=True)

## 9. Plots — day and night

Two colouring schemes for two questions:

- **`plot_pointcloud_3d`** — one panel per gene, black-to-hue ramp by intensity.
  Quantitative: *where is this gene on?*
- **`plot_additive_3d` / `plot_additive_2d`** — all genes at once; hue from the
  additive mix at full brightness, with size and opacity carrying intensity.
  Qualitative: *which combinations occur where?*

Separating hue from brightness is what keeps a three-colour overlay readable — a
nucleus expressing one dim gene still shows that gene's hue instead of fading out.

A nucleus counts as expressing when **one** channel clears threshold, not when the
channel sum does; otherwise a nucleus could be coloured while being positive for
nothing.

Every function takes `mode="dark"` or `mode="light"`, and the greys, outlines and
gain differ between them — colour that reads bright on black reads washed out on
white.

In [ ]:
wf.plot_all(modes=("dark", "light"))   # writes both themes to <cohort>/figures/

In [ ]:
from register_embryos import plot_additive_3d

# Interactive, in the notebook. Swap to mode="light" for the figure twin.
plot_additive_3d(wf.atlas.points, mode="dark", coords=("x", "y", "z"),
                 title=f"{wf.cohort.name} atlas")

In [ ]:
from register_embryos import plot_pointcloud_3d

plot_pointcloud_3d(wf.atlas.points, mode="dark", coords=("x", "y", "z"),
                   title=f"{wf.cohort.name} atlas — per gene")

## 10. Record what produced this

Inputs, parameters, orientations, versions. Worth having once several cohorts have
been run with different bin sizes, segmentation modes and `k` values.

In [ ]:
wf.save_manifest()
print()
for key, value in wf.outputs().summary().items():
    print(f"  {key}: {value}")

## 11. Comparing cohorts — align two atlases

Puts two cohorts (e.g. mutant vs wild type) in one coordinate frame so their
expression domains can be compared directly. `center_first=True` because atlases
from different cohorts can sit in quite different coordinate ranges.

In [ ]:
# from register_embryos import align_atlases
#
# aligned = align_atlases({"wt": wt_atlas, "pbx": pbx_atlas},
#                         reference_label="wt", center_first=True)
# print(aligned.stats.to_string(index=False))

## 12. Re-doing the cheap steps without the expensive ones

Registration and the atlas only need the nucleus table, so a different reference,
`k`, or embryo exclusion costs seconds — no images, no Cellpose:

```bash
register-embryos atlas out/wt_12s_dorsal_20X/combined_nucleus_table.csv \
    -o out/retry --k 6 --exclude 20260825_1.2_wt_12s_dorsal_20X_prdm1a_fli1a_tbx1
```

Or in Python:

In [ ]:
# import pandas as pd
# from register_embryos import build_atlas, register_cohort
#
# table = pd.read_csv(wf.output_dir / "combined_nucleus_table.csv")
# reg = register_cohort(table, reference_embryo_id=..., n_downsample=5000)
# atlas = build_atlas(reg.registered, reg.reference_embryo_id, k_neighbors=6)

## Project-specific extras

Steps that depend on the biology of a particular experiment live in
`register_embryos.contrib` and are **not** part of the standard workflow, so they
never get applied to a panel where their assumption does not hold:

```python
from register_embryos.contrib import midline_filter
clean, bounds = midline_filter(wf.atlas.points, marker="wt1a", axis="y")
```